# ML-08 — Capstone Modeling (Week 5)

Lane: **Refresh / Content Opportunity Scoring** — confirmed.

This notebook trains a first learned model for the lane and compares it, honestly, to my Week-4 rule baseline on the same data, the same client-holdout split, and the same metric (Precision@K). It ends with the model-vs-baseline table and a short error analysis. Everything runs locally on the starter slice (30,000 rows) — sklearn models train in seconds on CPU; no GPU is needed for this lane.


## 1. Method choice and why

My lane is a **ranking problem**: out of thousands of pages, which should a reviewer fix first. The observed label (`trend_direction == "down"`) makes it a binary classification underneath, but the decision needs **scores to rank**, not hard labels — so I train classifiers and rank by their predicted probability, evaluated at Precision@20/50/100 (matching the reviewer's capacity constraint).

Method menu, in order of complexity:

- **Logistic Regression** — the readable baseline model; its coefficients tell me *which way* each signal pushes.
- **Decision Tree** (depth-limited) — a printable rule, a sanity mirror of my Week-4 rule.
- **Random Forest** — the stronger, still-interpretable ensemble.
- **HistGradientBoosting** — the modern gradient-boosted option, included to check whether complexity actually buys anything.

Selection rule: the simplest model that is not meaningfully worse wins. The cell below builds the feature matrix from **safe features only** — no `trend_direction`, no `trend_pct`, no `*_last_30d` / `*_prev_30d` trend inputs, no IDs, no provider/model metadata.

> **Honesty note on features:** my first draft included `days_with_impressions` / `days_with_sessions`. I dropped both after finding `days_with_impressions` **alone** reaches ROC-AUC 0.758 on held-out clients — it is a same-window presence counter that nearly encodes the trend-label categories (pages with few impression-days are mostly `flat`; pages with many are `down`/`up`/`stable`). Keeping it would flatter the model without teaching it anything transferable.


In [1]:
# ---- Setup + feature matrix ----
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

print("sklearn", sklearn.__version__)

ROOT = Path(os.getcwd())
while not (ROOT / "data" / "raw" / "content_refresh_anonymized.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

df = pd.read_csv(ROOT / "data" / "raw" / "content_refresh_anonymized.csv")
df["declining"] = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)

# Safe numeric + categorical features (no trend columns, no 30-day comparison windows, no IDs)
NUMERIC = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
CATEGORICAL = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

num = df[NUMERIC].apply(pd.to_numeric, errors="coerce")
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "users_90d",
            "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d", "search_volume"]:
    num[f"log_{col}"] = np.log1p(num[col])
num["has_position"] = (df["avg_position"] > 0).astype(int)   # avg_position=0 means "no data"
num["has_word_count"] = (df["word_count"] > 0).astype(int)   # word_count blank follows content_type
num = num.replace([np.inf, -np.inf], np.nan).fillna(0)

cat = df[CATEGORICAL].fillna("unknown").astype(str)
cat_enc = pd.get_dummies(cat, prefix=CATEGORICAL, dtype=float)

X = pd.concat([num.reset_index(drop=True), cat_enc.reset_index(drop=True)], axis=1)
y = df["declining"].to_numpy()
print(f"feature matrix: {X.shape}  |  label positive rate: {y.mean():.3f}")


sklearn 1.9.0
feature matrix: (30000, 63)  |  label positive rate: 0.542


## 2. Split design

**Grouped by client, not by row.** Pages from the same client share a content template, a traffic scale, and an instrumentation profile — random row-splitting would let the model memorize a client's quirks and still look good on a row that belongs to a *seen* client. The honest test is: can the model score pages from clients it has **never trained on**?

That is also where my Week-4 rule falls apart: its demand-percentile term is calibrated on the clients it was built on, and it does not transfer. Holding out ~20% of clients (same seed 42 as the reference pipeline) makes that failure visible instead of hiding it.


In [2]:
# ---- Client-holdout split (grouped by client_id) ----
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

clients = df["client_id"].drop_duplicates().to_numpy()
shuffled = rng.permutation(clients)
n_test_clients = max(1, int(round(len(shuffled) * 0.2)))
test_clients = set(shuffled[:n_test_clients])
test_mask = df["client_id"].isin(test_clients).to_numpy()

X_train, X_test = X[~test_mask], X[test_mask]
y_train, y_test = y[~test_mask], y[test_mask]

print(f"clients: {len(clients)} total -> {n_test_clients} held out")
print(f"train rows: {len(y_train):,} (positive rate {y_train.mean():.3f})")
print(f"test  rows: {len(y_test):,} (positive rate {y_test.mean():.3f})")


clients: 32 total -> 6 held out
train rows: 27,675 (positive rate 0.555)
test  rows: 2,325 (positive rate 0.391)


## 3. Train + compare vs my baseline

Both the baseline rule and the models are evaluated **only on the held-out test clients**, with the same Precision@K metric. My Week-4 baseline is re-encoded exactly (`stale × demand × ctr_weak`) and its score computed the same way — only now on unseen clients.

> **Second honesty note:** in this starter slice the label is `trend_direction` over the *current* trailing window, so every 90-day feature overlaps the label period. The numbers below therefore measure "can the model recover the current decline state **and transfer it to unseen clients**" — not a true forward prediction. The forward test (prior window → next window) is the Week-6+ warehouse job. This week's job is to beat the baseline cleanly on the same, honest split.


In [3]:
# ---- Week-4 baseline rule, re-encoded exactly ----
def percentile_rank(series: pd.Series) -> pd.Series:
    return series.rank(method="average", pct=True).fillna(0)

stale = (df["days_since_last_update"] >= 90).astype(int)
demand = percentile_rank(np.log1p(df["impressions_90d"]))
ctr_weak = 1 - df["ctr"].clip(lower=0, upper=2) / 2.0
baseline_score = (stale * demand * ctr_weak).to_numpy()

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores, dtype=float))
    return float(np.asarray(labels)[order[:k]].mean())

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

models = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "decision_tree": DecisionTreeClassifier(
        max_depth=5, min_samples_leaf=50, class_weight="balanced", random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_leaf=25,
        class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE),
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.1, max_leaf_nodes=31, min_samples_leaf=50,
        l2_regularization=1.0, random_state=RANDOM_STATE),
}

def evaluate(name, test_scores):
    return {
        "model": name,
        "precision_at_20": round(float(precision_at_k(test_scores, y_test, 20)), 3),
        "precision_at_50": round(float(precision_at_k(test_scores, y_test, 50)), 3),
        "precision_at_100": round(float(precision_at_k(test_scores, y_test, 100)), 3),
        "roc_auc": round(float(roc_auc_score(y_test, test_scores)), 3),
        "average_precision": round(float(average_precision_score(y_test, test_scores)), 3),
    }

rows = [evaluate("baseline (W4 rule)", baseline_score[test_mask])]
trained = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    trained[name] = model
    rows.append(evaluate(name, proba))

comparison = pd.DataFrame(rows)
print(f"Base rate on test: {y_test.mean():.3f}  (a random ranking scores this at every K)")
print(comparison.to_string(index=False))

# ---- Write a metrics receipt (committed; regenerated on every run) ----
OUT_DIR = ROOT / "work" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
receipt = {
    "rows": int(len(df)),
    "train_rows": int(len(y_train)),
    "test_rows": int(len(y_test)),
    "test_clients": int(n_test_clients),
    "split": "client_holdout",
    "random_state": RANDOM_STATE,
    "test_base_rate": round(float(y_test.mean()), 4),
    "features_excluded": [
        "trend_direction", "trend_pct",
        "impressions_last_30d", "impressions_prev_30d",
        "clicks_last_30d", "clicks_prev_30d",
        "sessions_last_30d", "sessions_prev_30d",
        "days_with_impressions", "days_with_sessions",
    ],
    "comparison": comparison.to_dict(orient="records"),
}
(OUT_DIR / "w05_model_metrics.json").write_text(json.dumps(receipt, indent=2))
print(f"\nwrote {OUT_DIR / 'w05_model_metrics.json'}")


Base rate on test: 0.391  (a random ranking scores this at every K)
                 model  precision_at_20  precision_at_50  precision_at_100  roc_auc  average_precision
    baseline (W4 rule)              0.5             0.26              0.32    0.493              0.392
   logistic_regression              0.7             0.72              0.71    0.711              0.580
         decision_tree              0.6             0.56              0.60    0.735              0.572
         random_forest              0.8             0.66              0.65    0.735              0.584
hist_gradient_boosting              0.8             0.72              0.69    0.721              0.578

wrote /Users/egealgel/Documents/FlyRankAI/flyrank-ml-internship-starter/work/outputs/w05_model_metrics.json


## 4. Errors and interpretation

Three questions, in order: what does the model lean on, where is it wrong, and what do concrete mistakes look like. The cell below answers all three for the **logistic regression** — it ties HistGradientBoosting on Precision@50 while being far simpler, and its coefficients are directly readable, so it is the model I carry forward for interpretation.


In [4]:
# ---- 4a. What the model leans on (permutation importance on the test set) ----
from sklearn.inspection import permutation_importance

best_model = trained["logistic_regression"]
perm = permutation_importance(best_model, X_test, y_test, n_repeats=5,
                              scoring="roc_auc", random_state=RANDOM_STATE, n_jobs=-1)
importance = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
print("Permutation importance (test, ROC-AUC) — top 10:")
print(importance.head(10).round(4).to_string())

# ---- 4b. Direction of each signal (standardized LR coefficients) ----
coefs = pd.Series(best_model.named_steps["model"].coef_[0], index=X.columns)
print("\nHigher -> MORE likely declining (top 6):")
print(coefs.sort_values(ascending=False).head(6).round(3).to_string())
print("\nHigher -> LESS likely declining (top 6):")
print(coefs.sort_values().head(6).round(3).to_string())

# ---- 4c. Where the model is wrong ----
proba = best_model.predict_proba(X_test)[:, 1]
te = df[test_mask].copy()
te["proba"] = proba
te["pred"] = (proba >= 0.5).astype(int)
te["wrong"] = (te["pred"] != te["declining"]).astype(int)

print("\nError rate by content_type:")
print(te.groupby("content_type", observed=True)
      .agg(n=("content_id", "size"), error_rate=("wrong", "mean"), positive_rate=("declining", "mean"))
      .round(3).to_string())

print("\nError rate by position_tier:")
print(te.groupby("position_tier", observed=True)
      .agg(n=("content_id", "size"), error_rate=("wrong", "mean"), positive_rate=("declining", "mean"))
      .round(3).to_string())

# ---- 4d. Three concrete mistakes ----
show_cols = ["content_id", "proba", "impressions_90d", "ctr", "avg_position",
             "content_age_days", "word_count", "content_type", "trend_direction"]
fp = te[(te["proba"] >= 0.5) & (te["declining"] == 0)].sort_values("proba", ascending=False)
fn = te[(te["proba"] < 0.5) & (te["declining"] == 1)].sort_values("proba")
print("\nFalse positives (model confident, actually NOT declining):")
print(fp[show_cols].head(2).to_string(index=False))
print("\nFalse negative (model certain it is fine, actually declining):")
print(fn[show_cols].head(1).to_string(index=False))


Permutation importance (test, ROC-AUC) — top 10:
log_impressions_90d             0.1503
word_count                      0.1125
has_position                    0.0670
log_clicks_90d                  0.0342
content_age_days                0.0308
char_count                      0.0234
content_type_keyword article    0.0192
competition_level_unknown       0.0177
content_type_feedly article     0.0144
competition_level_LOW           0.0104

Higher -> MORE likely declining (top 6):
word_count               1.699
log_impressions_90d      1.256
has_position             0.830
log_scroll_events_90d    0.418
sessions_90d             0.375
impression_tier_low      0.196

Higher -> LESS likely declining (top 6):
char_count                 -1.425
log_clicks_90d             -0.665
users_90d                  -0.442
avg_position               -0.424
log_users_90d              -0.263
log_engaged_sessions_90d   -0.242

Error rate by content_type:
                    n  error_rate  positive_rate
content_t

**What this says.**

- **What it leans on:** `log_impressions_90d` (demand), `word_count`/`char_count` (depth), `has_position` (whether rank is measured), `log_clicks_90d` (engagement), and `content_age_days` — all plausibly related to decline, none suspiciously perfect once the presence-counters are out. *(Caveat: `word_count` and `char_count` are near-duplicates, so their individual coefficient signs are unstable — read them together as "length," not separately.)*
- **Direction:** more impressions and longer pages lean toward "declining" (they have traffic to lose), while more clicks/users/engagement and older age lean toward "healthy" — engaged pages keep their audience.
- **Where it's wrong:** the model is clean on `top_3` pages (error ~10%) but much weaker on `deep` pages (error ~62%) and on `keyword article` rows (error ~42%) — the ambiguous middle where the decline signal is noisiest.
- **The failure shape:** its false positives are long keyword articles with **zero clicks / zero CTR** that look like lost causes but are actually `new`/`up`/`stable`; its false negatives are pages with **healthy CTR** that the model is certain are fine yet are still losing impressions. The model over-trusts CTR/clicks as a proxy for health — exactly the structural bias my Week-4 CTR signal check flagged.

The headline, restated: on unseen clients the Week-4 rule ranks **below the base rate**, and every learned model beats it by a wide margin — logistic regression and gradient boosting lift Precision@50 from ~0.26 to ~0.72 on the same split. That is the bar next week's model must clear, and the errors above are the concrete places a better model earns its complexity.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (executed via nbconvert)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit the repo URL on the card. Done.
